# IMC Prosperity 4 — Round 3 Data Analysis & Alpha Discovery

This notebook provides **systematic, exhaustive analysis** of Round 3 market data. Every section is self-contained with interpretation guides so you know *what to look for* and *what action to take*.

## How to use
1. Drop your CSV files into the same directory as this notebook  
2. Update `DATA_DIR` and `ROUND` in **Section 0**  
3. Run all cells top-to-bottom  
4. Each section has a **"What to extract"** block telling you the trading signal

## Data format expected
- `prices_round_X_day_Y.csv` — order book snapshots (every 100ms)
- `trades_round_X_day_Y.csv` — public market trades

**Prices columns:** `day, timestamp, product, bid_price_1..3, bid_volume_1..3, ask_price_1..3, ask_volume_1..3, mid_price, profit_and_loss`  
**Trades columns:** `timestamp, buyer, seller, symbol, currency, price, quantity`

---
## Table of Contents
0. Configuration & Data Loading  
1. Data Overview & Quality Checks  
2. Price Dynamics (Mid Price, Returns, Volatility)  
3. Order Book Microstructure  
4. Market Trades Analysis & Bot Detection  
5. Basket Arbitrage Analysis (PICNIC_BASKETs)  
6. Options Pricing — Black-Scholes & Implied Volatility Smile  
7. Conversion / Tariff Products (MAGNIFICENT_MACARONS style)  
8. Mean-Reversion & Trend Signals  
9. Cross-Product Correlation & Lead-Lag  
10. Named Bot Behavior (Olivia, Vinnie, etc.)  
11. PnL Attribution & Position Analysis  
12. Alpha Signal Summary Dashboard  

---
## Section 0 — Configuration & Data Loading

In [ ]:
# ── INSTALL / IMPORT ──────────────────────────────────────────────────────────
import subprocess, sys
def pip(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for p in ['pandas', 'numpy', 'matplotlib', 'seaborn', 'scipy', 'statsmodels',
          'plotly', 'scikit-learn']:
    pip(p)

import warnings
warnings.filterwarnings('ignore')

import os, glob, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.optimize import brentq
from scipy.stats import norm
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, coint
from sklearn.linear_model import LinearRegression
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
})
PALETTE = sns.color_palette('tab10')

print('Libraries loaded ✓')

In [ ]:
# ── USER CONFIGURATION ────────────────────────────────────────────────────────
DATA_DIR = '.'          # folder containing your CSV files
ROUND    = 3            # which round number
DAYS     = [-2, -1, 0]  # Prosperity uses day -2, -1, 0 within each round

# ── PRODUCT METADATA ─────────────────────────────────────────────────────────
# Basket compositions (update if Round 3 changes these)
BASKET_COMPOSITIONS = {
    'PICNIC_BASKET1': {'CROISSANTS': 6, 'JAMS': 3, 'DJEMBES': 1},
    'PICNIC_BASKET2': {'CROISSANTS': 4, 'JAMS': 2},
}

# Voucher options: strike → product name
VOUCHER_STRIKES = {
    9500:  'VOLCANIC_ROCK_VOUCHER_9500',
    9750:  'VOLCANIC_ROCK_VOUCHER_9750',
    10000: 'VOLCANIC_ROCK_VOUCHER_10000',
    10250: 'VOLCANIC_ROCK_VOUCHER_10250',
    10500: 'VOLCANIC_ROCK_VOUCHER_10500',
}
UNDERLYING = 'VOLCANIC_ROCK'          # underlying for options
DAYS_TO_EXPIRY_START = 7              # number of trading days until expiry at round start
TICKS_PER_DAY = 10_000               # timestamps per simulated day (0..999_900 step 100)

# Conversion product (Macarons style)
CONVERSION_PRODUCT = 'MAGNIFICENT_MACARONS'
CONVERSION_LIMIT   = 10              # max conversions per step

# Known named bots in the market_trades data
KNOWN_BOTS = ['Olivia', 'Vinnie', 'Pablo', 'Caesar', 'Raj', 'Valentina']

print(f'Config: Round {ROUND}, Days {DAYS}')

In [ ]:
# ── DATA LOADING ─────────────────────────────────────────────────────────────
def load_prices(data_dir, round_n, days):
    frames = []
    for day in days:
        # try standard naming: prices_round_X_day_Y.csv (Y may be negative)
        patterns = [
            f'{data_dir}/prices_round_{round_n}_day_{day}.csv',
            f'{data_dir}/prices_round_{round_n}_day_{day:+d}.csv',
        ]
        for pat in patterns:
            matches = glob.glob(pat)
            if matches:
                df = pd.read_csv(matches[0], sep=';')
                df['source_day'] = day
                frames.append(df)
                print(f'  Loaded prices: {matches[0]}  ({len(df):,} rows)')
                break
        else:
            print(f'  [WARN] No prices file for round={round_n} day={day}')
    if not frames:
        print('[ERROR] No price files found. Check DATA_DIR and ROUND.')
        return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True)
    df.columns = df.columns.str.strip().str.lower()
    # Unified timestamp across days
    df['abs_ts'] = df['source_day'] * TICKS_PER_DAY * 100 + df['timestamp']
    return df


def load_trades(data_dir, round_n, days):
    frames = []
    for day in days:
        patterns = [
            f'{data_dir}/trades_round_{round_n}_day_{day}.csv',
            f'{data_dir}/trades_round_{round_n}_day_{day:+d}.csv',
        ]
        for pat in patterns:
            matches = glob.glob(pat)
            if matches:
                df = pd.read_csv(matches[0], sep=';')
                df['source_day'] = day
                frames.append(df)
                print(f'  Loaded trades: {matches[0]}  ({len(df):,} rows)')
                break
        else:
            print(f'  [WARN] No trades file for round={round_n} day={day}')
    if not frames:
        return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True)
    df.columns = df.columns.str.strip().str.lower()
    df['abs_ts'] = df['source_day'] * TICKS_PER_DAY * 100 + df['timestamp']
    return df


print('Loading data...')
prices_raw = load_prices(DATA_DIR, ROUND, DAYS)
trades_raw = load_trades(DATA_DIR, ROUND, DAYS)

PRODUCTS = sorted(prices_raw['product'].unique()) if not prices_raw.empty else []
print(f'\nProducts detected: {PRODUCTS}')

---
## Section 1 — Data Overview & Quality Checks

**What to check:** Missing timestamps, gaps in order book data, products with thin books. Any product with >5% missing mid-prices has data quality issues that may affect signal reliability.

In [ ]:
# ── OVERVIEW TABLE ────────────────────────────────────────────────────────────
if not prices_raw.empty:
    summary = prices_raw.groupby('product').agg(
        rows            = ('timestamp', 'count'),
        ts_min          = ('timestamp', 'min'),
        ts_max          = ('timestamp', 'max'),
        mid_price_mean  = ('mid_price', 'mean'),
        mid_price_std   = ('mid_price', 'std'),
        mid_price_min   = ('mid_price', 'min'),
        mid_price_max   = ('mid_price', 'max'),
        pnl_final       = ('profit_and_loss', 'last'),
    ).round(2)
    summary['missing_mid%'] = (prices_raw.groupby('product')['mid_price'].apply(
        lambda x: x.isna().mean() * 100)).round(1)
    print('=== Price Data Summary ===')
    display(summary)

if not trades_raw.empty:
    t_summary = trades_raw.groupby('symbol' if 'symbol' in trades_raw.columns else 'product').agg(
        trade_count = ('price', 'count'),
        total_vol   = ('quantity', 'sum'),
        price_mean  = ('price', 'mean'),
        price_std   = ('price', 'std'),
    ).round(2)
    print('\n=== Trade Data Summary ===')
    display(t_summary)

In [ ]:
# ── TIMESTAMP CONTINUITY ─────────────────────────────────────────────────────
# Gaps indicate missed data; clusters indicate heavy-traffic periods
if not prices_raw.empty:
    fig, axes = plt.subplots(len(DAYS), 1, figsize=(14, 3 * len(DAYS)), sharex=False)
    if len(DAYS) == 1:
        axes = [axes]
    for ax, day in zip(axes, DAYS):
        day_data = prices_raw[prices_raw['source_day'] == day]
        for prod in PRODUCTS:
            sub = day_data[day_data['product'] == prod]
            if sub.empty:
                continue
            gaps = sub['timestamp'].diff()
            ax.plot(sub['timestamp'], gaps, label=prod, alpha=0.6, linewidth=0.8)
        ax.set_title(f'Day {day} — Timestamp Gaps Between Rows (should be 100)')
        ax.set_ylabel('Gap (ticks)')
        ax.legend(fontsize=7, ncol=3)
    plt.tight_layout()
    plt.suptitle('Data Continuity Check', y=1.01, fontsize=12, fontweight='bold')
    plt.show()
    print('\n>> Interpretation: Gaps > 100 mean missing ticks. Large spikes reveal data outages.\n'
          '>> If gaps cluster at the same timestamp across products, it is systemic (server-side).')

---
## Section 2 — Price Dynamics

**What to extract:**
- **Flat/mean-reverting** mid-price → market-making opportunities (fade extremes)
- **Trending** mid-price → momentum / directional strategies  
- **High-volatility bursts** → reduce position size during chaos, or exploit mean reversion after  
- **Rolling correlation between returns** → detect lead-lag or cointegration candidates

In [ ]:
# ── MID-PRICE TIME SERIES ─────────────────────────────────────────────────────
if not prices_raw.empty:
    n_prod = len(PRODUCTS)
    n_cols = min(2, n_prod)
    n_rows = (n_prod + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 3.5 * n_rows))
    axes = np.array(axes).flatten()

    for i, prod in enumerate(PRODUCTS):
        sub = prices_raw[prices_raw['product'] == prod].sort_values('abs_ts')
        ax  = axes[i]
        for day in DAYS:
            d = sub[sub['source_day'] == day]
            ax.plot(d['timestamp'], d['mid_price'], label=f'Day {day}', linewidth=0.9)
        ax.set_title(prod, fontweight='bold')
        ax.set_xlabel('Timestamp')
        ax.set_ylabel('Mid Price')
        ax.legend(fontsize=7)
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    plt.suptitle('Mid-Price Time Series — All Products All Days', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print('>> Look for: clear drifts (trend-follow), flat ranges (market-make), sharp spikes (bot events)')

In [ ]:
# ── RETURN DISTRIBUTIONS ──────────────────────────────────────────────────────
# Tick-by-tick log returns; fat tails mean price jumps are common → adjust position sizing
if not prices_raw.empty:
    fig, axes = plt.subplots(2, max(1, len(PRODUCTS) // 2 + len(PRODUCTS) % 2),
                              figsize=(14, 7))
    axes = np.array(axes).flatten()
    ret_stats = []

    for i, prod in enumerate(PRODUCTS):
        sub = prices_raw[prices_raw['product'] == prod].sort_values('abs_ts')
        ret = np.log(sub['mid_price'] / sub['mid_price'].shift(1)).dropna()

        mu, sigma = ret.mean(), ret.std()
        skew = ret.skew()
        kurt = ret.kurtosis()
        ret_stats.append({'product': prod, 'mean_return': mu, 'std': sigma,
                           'skewness': skew, 'excess_kurtosis': kurt,
                           'annualised_vol': sigma * np.sqrt(TICKS_PER_DAY)})

        if i < len(axes):
            ax = axes[i]
            ax.hist(ret, bins=80, density=True, alpha=0.65, color=PALETTE[i % 10],
                    label='Empirical')
            # Overlay normal distribution
            x = np.linspace(ret.min(), ret.max(), 300)
            ax.plot(x, norm.pdf(x, mu, sigma), 'k--', linewidth=1.5, label='Normal fit')
            ax.set_title(f'{prod}\nKurt={kurt:.1f}  Skew={skew:.2f}', fontsize=8)
            ax.legend(fontsize=6)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    plt.suptitle('Tick Return Distributions (log returns)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    ret_stats_df = pd.DataFrame(ret_stats).set_index('product')
    display(ret_stats_df.round(6))
    print('\n>> Excess kurtosis > 3 means fat tails (price jumps). High skew = asymmetric risk.\n'
          '>> Mean return ≠ 0 (consistently) may indicate a persistent bot pushing prices directionally.')

In [ ]:
# ── ROLLING VOLATILITY ────────────────────────────────────────────────────────
# Shows when market is calm vs. turbulent; useful for sizing and avoiding MM during spikes
ROLL_WINDOW = 200  # ticks (~20 seconds at 100ms)

if not prices_raw.empty:
    fig, ax = plt.subplots(figsize=(14, 4))
    for i, prod in enumerate(PRODUCTS):
        sub = prices_raw[prices_raw['product'] == prod].sort_values('abs_ts')
        ret = np.log(sub['mid_price'] / sub['mid_price'].shift(1))
        vol = ret.rolling(ROLL_WINDOW).std() * np.sqrt(TICKS_PER_DAY)
        ax.plot(sub['abs_ts'], vol, label=prod, alpha=0.75, linewidth=0.9)
    ax.set_title(f'Rolling {ROLL_WINDOW}-tick Annualised Volatility', fontweight='bold')
    ax.set_xlabel('Absolute Timestamp')
    ax.set_ylabel('Annualised Vol')
    ax.legend(ncol=3, fontsize=8)
    plt.tight_layout()
    plt.show()
    print('>> Periods of high vol: widen your quotes or reduce position size to avoid adverse selection.\n'
          '>> Sudden vol spikes often coincide with bot large orders — look for systematic patterns by day.')

---
## Section 3 — Order Book Microstructure

**What to extract:**
- **Bid-ask spread** → baseline cost of a round-trip trade; your edge must exceed this  
- **Order book imbalance (OBI)** → short-term directional predictor (classic HFT signal)  
- **Depth at each level** → where the market has conviction  
- **Spread regime** → periods of tight vs. wide spread inform when to be aggressive vs. passive

In [ ]:
# ── BID-ASK SPREAD ANALYSIS ───────────────────────────────────────────────────
if not prices_raw.empty:
    spread_stats = []
    fig, axes = plt.subplots(len(PRODUCTS), 2, figsize=(14, 3 * len(PRODUCTS)))
    if len(PRODUCTS) == 1:
        axes = [axes]

    for i, prod in enumerate(PRODUCTS):
        sub = prices_raw[prices_raw['product'] == prod].copy()
        sub['spread'] = sub['ask_price_1'] - sub['bid_price_1']

        spread_stats.append({
            'product'   : prod,
            'mean_spread': sub['spread'].mean(),
            'median_spread': sub['spread'].median(),
            'pct_spread': (sub['spread'] / sub['mid_price']).mean() * 100,
            'min_spread': sub['spread'].min(),
            'max_spread': sub['spread'].max(),
        })

        ax_ts, ax_hist = axes[i][0], axes[i][1]
        ax_ts.plot(sub['abs_ts'], sub['spread'], linewidth=0.6, color=PALETTE[i % 10])
        ax_ts.set_title(f'{prod} — Spread over Time', fontsize=9)
        ax_ts.set_ylabel('Spread')

        ax_hist.hist(sub['spread'].dropna(), bins=40, color=PALETTE[i % 10], alpha=0.75)
        ax_hist.set_title(f'{prod} — Spread Distribution', fontsize=9)
        ax_hist.axvline(sub['spread'].median(), color='red', linestyle='--', label='Median')
        ax_hist.legend(fontsize=7)

    plt.suptitle('Bid-Ask Spread Analysis', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    display(pd.DataFrame(spread_stats).set_index('product').round(4))
    print('\n>> Your profitable trade needs edge > half the spread (cost to cross).\n'
          '>> Products with median spread = 1 tick are very competitive — require precision signals.\n'
          '>> Wide spreads = more room for passive MM profit per fill.')

In [ ]:
# ── ORDER BOOK IMBALANCE (OBI) SIGNAL ─────────────────────────────────────────
# OBI = (bid_vol - ask_vol) / (bid_vol + ask_vol)
# Positive OBI → more buyers → price likely to go UP
# This is one of the most reliable ultra-short-term price predictors in HFT literature

if not prices_raw.empty and all(c in prices_raw.columns for c in
        ['bid_volume_1','ask_volume_1','bid_volume_2','ask_volume_2']):

    FORWARD_TICKS = 10  # predict price change N ticks ahead

    fig, axes = plt.subplots(len(PRODUCTS), 1, figsize=(14, 4 * len(PRODUCTS)))
    if len(PRODUCTS) == 1:
        axes = [axes]

    obi_results = []
    for i, prod in enumerate(PRODUCTS):
        sub = prices_raw[prices_raw['product'] == prod].copy().sort_values('abs_ts').reset_index(drop=True)

        # L1 OBI
        bv = sub['bid_volume_1'].abs()
        av = sub['ask_volume_1'].abs()
        sub['obi_l1'] = (bv - av) / (bv + av).replace(0, np.nan)

        # L2 OBI (weighted)
        if 'bid_volume_2' in sub.columns:
            bv2 = sub['bid_volume_2'].abs()
            av2 = sub['ask_volume_2'].abs()
            sub['obi_l2'] = (bv + bv2 - av - av2) / (bv + bv2 + av + av2).replace(0, np.nan)

        # Forward return
        sub['fwd_ret'] = sub['mid_price'].shift(-FORWARD_TICKS) / sub['mid_price'] - 1

        # Correlation OBI → fwd return
        corr = sub[['obi_l1', 'fwd_ret']].dropna().corr().iloc[0, 1]
        obi_results.append({'product': prod, f'obi_l1_vs_fwd{FORWARD_TICKS}tick_ret_corr': corr})

        ax = axes[i]
        ax2 = ax.twinx()
        ax.plot(sub['abs_ts'], sub['obi_l1'], color='steelblue', alpha=0.5, linewidth=0.6, label='OBI L1')
        ax2.plot(sub['abs_ts'], sub['mid_price'], color='orange', linewidth=0.9, label='Mid Price')
        ax.set_title(f'{prod} — OBI L1 vs Mid Price  (fwd corr = {corr:.3f})', fontweight='bold')
        ax.set_ylabel('OBI (-1=full ask, +1=full bid)', color='steelblue')
        ax2.set_ylabel('Mid Price', color='orange')
        ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(lines1 + lines2, labels1 + labels2, fontsize=7, loc='upper left')

    plt.suptitle('Order Book Imbalance (OBI) Signal', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    display(pd.DataFrame(obi_results).set_index('product').round(4))
    print(f'\n>> OBI-to-return correlation > 0.05 is actionable. > 0.10 is strong.\n'
          f'>> If correlation is consistently positive: buy when OBI > threshold, sell when OBI < -threshold.\n'
          f'>> Use {FORWARD_TICKS}-tick forward window — tune this per product.')

In [ ]:
# ── ORDER BOOK DEPTH HEATMAP ──────────────────────────────────────────────────
# Shows where liquidity pools; consistent large bids/asks = "wall" that price respects
if not prices_raw.empty:
    for prod in PRODUCTS:
        sub = prices_raw[prices_raw['product'] == prod].sort_values('abs_ts')
        if sub.empty:
            continue

        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        # Sample every 50 rows to keep plot readable
        sub_s = sub.iloc[::max(1, len(sub)//500)]

        for j, side in enumerate(['bid', 'ask']):
            price_cols  = [f'{side}_price_{k}'  for k in range(1, 4)]
            vol_cols    = [f'{side}_volume_{k}' for k in range(1, 4)]
            available_p = [c for c in price_cols if c in sub_s.columns]
            available_v = [c for c in vol_cols   if c in sub_s.columns]

            ax = axes[j]
            for k, (pc, vc) in enumerate(zip(available_p, available_v)):
                sc = ax.scatter(sub_s['abs_ts'], sub_s[pc],
                                c=sub_s[vc].abs(), cmap='YlOrRd',
                                s=2, alpha=0.7, vmin=0, vmax=sub_s[vc].abs().quantile(0.95))
            plt.colorbar(sc, ax=ax, label='Volume')
            ax.set_title(f'{prod} — {side.capitalize()} Depth (colour = volume)', fontweight='bold')
            ax.set_xlabel('Timestamp')
            ax.set_ylabel('Price')

        plt.suptitle(f'Order Book Depth — {prod}', fontsize=11)
        plt.tight_layout()
        plt.show()

    print('>> Bright clusters (high volume) at consistent price levels = strong support/resistance.\n'
          '>> These levels are likely bot-driven walls — price tends to bounce back at them.')

---
## Section 4 — Market Trades Analysis & Bot Detection

**What to extract:**
- **Who is trading and how often** → named bots (Olivia, Vinnie, etc.) have predictable patterns  
- **Trade size distribution** → large consistent sizes = algorithmic counterparty  
- **Price impact of trades** → measure how much prices move after large trades (market impact curve)

In [ ]:
# ── TRADE FLOW OVERVIEW ───────────────────────────────────────────────────────
if not trades_raw.empty:
    symbol_col = 'symbol' if 'symbol' in trades_raw.columns else 'product'
    TRADE_PRODUCTS = trades_raw[symbol_col].unique().tolist()

    # Volume by product
    vol_by_prod = trades_raw.groupby(symbol_col)['quantity'].agg(['sum', 'count', 'mean'])
    vol_by_prod.columns = ['total_volume', 'trade_count', 'avg_size']

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    vol_by_prod['total_volume'].sort_values().plot.barh(ax=ax1, color=PALETTE[:len(TRADE_PRODUCTS)])
    ax1.set_title('Total Traded Volume by Product', fontweight='bold')
    ax1.set_xlabel('Cumulative Quantity')

    vol_by_prod['trade_count'].sort_values().plot.barh(ax=ax2, color=PALETTE[:len(TRADE_PRODUCTS)])
    ax2.set_title('Number of Trades by Product', fontweight='bold')
    ax2.set_xlabel('Trade Count')
    plt.tight_layout()
    plt.show()

    display(vol_by_prod.round(2))
    print('\n>> Products with high trade count but low avg size = retail/noise flow.\n'
          '>> Products with low count but high avg size = bot block trades — worth tracking.')

In [ ]:
# ── NAMED BOT ANALYSIS ────────────────────────────────────────────────────────
# Prosperity bots (Olivia, Vinnie, etc.) appear as buyer/seller in trades data.
# Their behaviour is deterministic across days — exploit it!

if not trades_raw.empty and 'buyer' in trades_raw.columns:
    # Identify all participants
    all_buyers  = trades_raw['buyer'].dropna().unique().tolist()
    all_sellers = trades_raw['seller'].dropna().unique().tolist()
    all_participants = list(set(all_buyers + all_sellers))
    print('All participants in trade data:', all_participants)

    bots_found = [b for b in KNOWN_BOTS if b in all_participants]
    print(f'Known bots found: {bots_found}')

    symbol_col = 'symbol' if 'symbol' in trades_raw.columns else 'product'

    for bot in bots_found:
        bot_buys  = trades_raw[trades_raw['buyer']  == bot]
        bot_sells = trades_raw[trades_raw['seller'] == bot]

        bot_all = pd.concat([
            bot_buys.assign(side='BUY'),
            bot_sells.assign(side='SELL')
        ])

        if bot_all.empty:
            continue

        prods = bot_all[symbol_col].unique()
        fig, axes = plt.subplots(len(prods), 2, figsize=(14, 4 * len(prods)))
        if len(prods) == 1:
            axes = [axes]

        for j, prod in enumerate(prods):
            pd_prod = bot_all[bot_all[symbol_col] == prod]
            buys  = pd_prod[pd_prod['side'] == 'BUY']
            sells = pd_prod[pd_prod['side'] == 'SELL']

            # Price of mid at time of trade
            price_sub = prices_raw[prices_raw['product'] == prod][['abs_ts', 'mid_price']]

            ax_ts, ax_hist = axes[j][0], axes[j][1]
            if not price_sub.empty:
                ax_ts.plot(price_sub['abs_ts'], price_sub['mid_price'],
                           color='grey', alpha=0.5, linewidth=0.7, label='Mid')
            ax_ts.scatter(buys['abs_ts'],  buys['price'],  color='green', s=20,
                          marker='^', zorder=5, label=f'{bot} BUY')
            ax_ts.scatter(sells['abs_ts'], sells['price'], color='red', s=20,
                          marker='v', zorder=5, label=f'{bot} SELL')
            ax_ts.set_title(f'{bot} — {prod} — Trades on Price Chart', fontweight='bold')
            ax_ts.legend(fontsize=7)

            # Size distribution
            ax_hist.hist(buys['quantity'].abs(),  bins=20, alpha=0.6, color='green', label='Buy qty')
            ax_hist.hist(sells['quantity'].abs(), bins=20, alpha=0.6, color='red',   label='Sell qty')
            ax_hist.set_title(f'{bot} — {prod} — Order Size Distribution', fontweight='bold')
            ax_hist.legend()

        plt.suptitle(f'Bot: {bot}', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()

    print('\n>> If a bot consistently buys at daily lows and sells at daily highs (like Olivia in Prosperity 3),\n'
          '>> track its historical pattern and fade/follow it. The bot is deterministic across days!')

In [ ]:
# ── TRADE PRICE IMPACT ────────────────────────────────────────────────────────
# Market impact: how much does a large trade move the mid price?
# If impact is predictable, you can front-run large trades

if not trades_raw.empty and not prices_raw.empty:
    symbol_col = 'symbol' if 'symbol' in trades_raw.columns else 'product'
    IMPACT_HORIZONS = [1, 5, 10, 50]  # ticks to measure price move after trade

    for prod in TRADE_PRODUCTS:
        t = trades_raw[trades_raw[symbol_col] == prod].copy()
        p = prices_raw[prices_raw['product'] == prod][['abs_ts','mid_price']].sort_values('abs_ts')
        if t.empty or p.empty:
            continue

        # Merge nearest mid price to each trade
        t_sorted = t.sort_values('abs_ts')
        p_idx    = p.set_index('abs_ts')['mid_price']

        results = []
        for _, row in t_sorted.iterrows():
            ts = row['abs_ts']
            # Price before trade
            pre = p_idx[:ts].iloc[-1] if not p_idx[:ts].empty else np.nan
            row_res = {'qty': row['quantity'], 'pre_price': pre}
            for h in IMPACT_HORIZONS:
                future_ts = ts + h * 100
                post = p_idx[ts:future_ts+1].iloc[-1] if not p_idx[ts:future_ts+1].empty else np.nan
                row_res[f'impact_{h}t'] = (post - pre) / pre if pre and not np.isnan(post) else np.nan
            results.append(row_res)

        imp_df = pd.DataFrame(results).dropna()
        if imp_df.empty:
            continue

        # Bin by trade size
        imp_df['qty_bin'] = pd.qcut(imp_df['qty'].abs(), q=5, duplicates='drop')
        impact_by_size = imp_df.groupby('qty_bin')[[f'impact_{h}t' for h in IMPACT_HORIZONS]].mean()

        fig, ax = plt.subplots(figsize=(10, 4))
        impact_by_size.plot(ax=ax, marker='o')
        ax.set_title(f'{prod} — Market Impact by Trade Size', fontweight='bold')
        ax.set_xlabel('Trade Size Quintile')
        ax.set_ylabel('Avg Return After Trade')
        ax.legend(title='Horizon', fontsize=8)
        plt.tight_layout()
        plt.show()

    print('\n>> If large trades (top quintile) consistently produce positive future returns → momentum signal.\n'
          '>> If negative → mean reversion; fade large trades by posting limit orders on the other side.')

---
## Section 5 — Basket Arbitrage Analysis

**What to extract:**  
The basket (e.g. PICNIC_BASKET1 = 6×CROISSANTS + 3×JAMS + 1×DJEMBE) should trade at a fair price equal to its components. When the spread between basket and fair value exceeds transaction costs, a risk-free arb exists.

- **Mean-reverting spread** → confirmed arb; trade the spread
- **Trending spread** → the basket has a premium/discount that the market accepts; adapt your fair-value model

In [ ]:
# ── BASKET FAIR VALUE & SPREAD ─────────────────────────────────────────────────
if not prices_raw.empty:
    for basket, comp in BASKET_COMPOSITIONS.items():
        basket_sub = prices_raw[prices_raw['product'] == basket][['abs_ts','mid_price']]
        if basket_sub.empty:
            print(f'[SKIP] {basket} not in price data')
            continue

        # Build synthetic fair value from components
        fv = basket_sub.set_index('abs_ts')['mid_price'].rename('basket_mid')
        synthetic = pd.Series(0.0, index=fv.index, name='synthetic')

        all_comps_found = True
        for comp_prod, qty in comp.items():
            comp_sub = prices_raw[prices_raw['product'] == comp_prod][['abs_ts','mid_price']]
            if comp_sub.empty:
                print(f'  [WARN] Component {comp_prod} missing from data')
                all_comps_found = False
                break
            comp_reindexed = comp_sub.set_index('abs_ts')['mid_price'].reindex(fv.index, method='ffill')
            synthetic = synthetic + qty * comp_reindexed

        if not all_comps_found:
            continue

        spread = fv - synthetic
        spread_df = pd.DataFrame({'basket': fv, 'synthetic': synthetic, 'spread': spread})

        fig, axes = plt.subplots(3, 1, figsize=(14, 10))

        axes[0].plot(spread_df.index, spread_df['basket'],    label='Basket Mid',    linewidth=0.9)
        axes[0].plot(spread_df.index, spread_df['synthetic'], label='Synthetic FV',  linewidth=0.9, linestyle='--')
        axes[0].set_title(f'{basket} — Basket vs Synthetic Fair Value', fontweight='bold')
        axes[0].legend()

        axes[1].plot(spread_df.index, spread_df['spread'], color='purple', linewidth=0.9)
        axes[1].axhline(spread_df['spread'].mean(),    color='red',   linestyle='--', label='Mean')
        axes[1].axhline(spread_df['spread'].mean() + 2*spread_df['spread'].std(),
                        color='orange', linestyle=':', label='+2σ')
        axes[1].axhline(spread_df['spread'].mean() - 2*spread_df['spread'].std(),
                        color='orange', linestyle=':', label='-2σ')
        axes[1].set_title(f'Basket−Synthetic Spread (mean={spread_df["spread"].mean():.2f})', fontweight='bold')
        axes[1].set_ylabel('Spread')
        axes[1].legend()

        # Spread histogram
        axes[2].hist(spread_df['spread'].dropna(), bins=60, color='purple', alpha=0.7)
        axes[2].axvline(0, color='black', linestyle='--', linewidth=1.5, label='Zero')
        axes[2].set_title('Spread Distribution', fontweight='bold')
        axes[2].legend()

        plt.suptitle(f'Basket Arbitrage — {basket}', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()

        # Stationarity test
        adf_stat, adf_p, *_ = adfuller(spread_df['spread'].dropna())
        print(f'\n{basket} Spread — ADF Statistic: {adf_stat:.4f}, p-value: {adf_p:.4f}')
        if adf_p < 0.05:
            print(f'  → STATIONARY spread (p < 0.05): confirmed mean-reversion arb. Trade spread!\n'
                  f'  → Entry at ±2σ, exit at mean. Size = min(basket limit, component limit / weight).')
        else:
            print(f'  → NON-STATIONARY spread: price in the basket\'s permanent discount/premium.\n'
                  f'  → Use rolling mean as fair value; still trade but with wider bands.')

In [ ]:
# ── COMPONENT CORRELATION MATRIX ──────────────────────────────────────────────
if not prices_raw.empty:
    # Pivot all products to columns, compute return correlations
    pivot = prices_raw.pivot_table(index='abs_ts', columns='product', values='mid_price', aggfunc='last')
    ret_pivot = np.log(pivot / pivot.shift(1)).dropna()

    corr_matrix = ret_pivot.corr()

    fig, ax = plt.subplots(figsize=(max(8, len(PRODUCTS)), max(6, len(PRODUCTS) - 1)))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
                ax=ax, mask=False, linewidths=0.5, vmin=-1, vmax=1)
    ax.set_title('Return Correlation Matrix — All Products', fontweight='bold', fontsize=12)
    plt.tight_layout()
    plt.show()
    print('\n>> Pairs with |correlation| > 0.7: strong co-movement → pairs trade candidate.\n'
          '>> Basket components should show correlation to basket. If they do not, decomposition is wrong.')

---
## Section 6 — Options Pricing: Black-Scholes & Implied Volatility Smile

**Context:** VOLCANIC_ROCK_VOUCHER products are European call options on VOLCANIC_ROCK. The market prices them — your job is to find where market IV differs from a consistent smile, then buy cheap or sell expensive options.

**What to extract:**
- **IV smile shape** → tells you which strikes are mispriced relative to the rest of the curve  
- **IV drift over time** → as expiry approaches, IV should stay stable; if it collapses, options are cheap  
- **Delta-hedging** → if an option is cheap in IV terms, buy it and delta-hedge the underlying

In [ ]:
# ── BLACK-SCHOLES UTILITIES ────────────────────────────────────────────────────
def bs_call_price(S, K, T, sigma, r=0.0):
    """European call option price via Black-Scholes."""
    if T <= 0 or sigma <= 0:
        return max(S - K, 0.0)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)


def implied_vol(market_price, S, K, T, r=0.0, tol=1e-6):
    """Implied volatility via Brent's method; returns NaN if no solution."""
    intrinsic = max(S - K, 0.0)
    if market_price <= intrinsic + tol:
        return np.nan
    try:
        f = lambda sigma: bs_call_price(S, K, T, sigma, r) - market_price
        return brentq(f, 1e-4, 10.0, xtol=tol)
    except Exception:
        return np.nan


def bs_delta(S, K, T, sigma, r=0.0):
    if T <= 0 or sigma <= 0:
        return 1.0 if S > K else 0.0
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return norm.cdf(d1)


print('Black-Scholes functions defined ✓')

In [ ]:
# ── IV SMILE ACROSS STRIKES ────────────────────────────────────────────────────
# For each timestamp: compute IV for each voucher strike, plot the smile curve

if not prices_raw.empty and UNDERLYING in prices_raw['product'].values:
    # Check which vouchers are in data
    vouchers_in_data = {k: v for k, v in VOUCHER_STRIKES.items() if v in prices_raw['product'].values}
    print(f'Voucher products found: {list(vouchers_in_data.values())}')

    if vouchers_in_data:
        underlying_p = prices_raw[prices_raw['product'] == UNDERLYING][['abs_ts','mid_price']].sort_values('abs_ts')

        # Compute IV at each tick for each voucher
        iv_frames = []
        for strike, voucher_name in vouchers_in_data.items():
            v_sub = prices_raw[prices_raw['product'] == voucher_name][['abs_ts','mid_price']].sort_values('abs_ts')
            merged = pd.merge_asof(v_sub, underlying_p, on='abs_ts', suffixes=('_opt', '_und'))
            merged['strike'] = strike

            # Time to expiry in years (1 trading day = 1/252 year; Prosperity uses ~7-day rounds)
            # abs_ts increases by 100 per tick; TICKS_PER_DAY ticks per day
            merged['elapsed_ticks'] = merged['abs_ts'] - merged['abs_ts'].min()
            total_ticks = DAYS_TO_EXPIRY_START * TICKS_PER_DAY * 100
            merged['T'] = (total_ticks - merged['elapsed_ticks'].clip(0, total_ticks)) / (total_ticks)
            merged['T_years'] = merged['T'] * DAYS_TO_EXPIRY_START / 252  # rough calendar

            # Compute IV row by row (downsampled for speed)
            merged = merged.iloc[::10].copy()
            merged['iv'] = merged.apply(
                lambda r: implied_vol(r['mid_price_opt'], r['mid_price_und'],
                                       r['strike'], max(r['T_years'], 1e-4)),
                axis=1
            )
            iv_frames.append(merged[['abs_ts','strike','mid_price_opt','mid_price_und','T','iv']])

        iv_df = pd.concat(iv_frames).dropna(subset=['iv'])
        iv_df = iv_df[(iv_df['iv'] > 0.01) & (iv_df['iv'] < 5.0)]  # sanity filter

        # ── IV SMILE SNAPSHOT (time-averaged) ────────────────────────────────
        smile = iv_df.groupby('strike')['iv'].agg(['mean','std','median'])

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        axes[0].errorbar(smile.index, smile['mean'], yerr=smile['std'],
                         marker='o', capsize=4, linewidth=2, color='royalblue', label='Mean IV ± 1σ')
        axes[0].plot(smile.index, smile['median'], marker='s', linestyle='--',
                     color='darkorange', label='Median IV')
        axes[0].set_title('Implied Volatility Smile — Time-Averaged', fontweight='bold')
        axes[0].set_xlabel('Strike Price')
        axes[0].set_ylabel('Implied Volatility')
        axes[0].legend()

        # IV over time for each strike
        for strike in sorted(vouchers_in_data.keys()):
            sub = iv_df[iv_df['strike'] == strike]
            axes[1].plot(sub['abs_ts'], sub['iv'], label=f'K={strike}', linewidth=0.9, alpha=0.8)
        axes[1].set_title('Implied Volatility Over Time by Strike', fontweight='bold')
        axes[1].set_xlabel('Timestamp')
        axes[1].set_ylabel('IV')
        axes[1].legend(fontsize=7)

        plt.suptitle('Options Implied Volatility Analysis', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()
        display(smile.round(4))

        print('\n>> The IV smile: if one strike has materially lower IV than the fitted curve → buy that option.\n'
          '>> If IV is collapsing towards expiry faster than theoretical theta decay → sell options (collect vol).\n'
          '>> The smile slope (skew) = market\'s assessment of downside risk. A steep put skew in a call =\n'
          '   the market expects asymmetric downside. Trade the skew by buying OTM calls + selling near-ATM.')
    else:
        print('No voucher products found in data. Skip this section if options are not in Round 3.')
else:
    print(f'{UNDERLYING} not found in data. Skipping options analysis.')

In [ ]:
# ── BLACK-SCHOLES FAIR VALUE vs MARKET PRICE ──────────────────────────────────
# When BS price (using fitted IV) diverges from market price → mispriced option

if not prices_raw.empty and UNDERLYING in prices_raw['product'].values and 'iv_df' in dir():
    # Use median IV across all strikes as volatility input
    median_iv = iv_df['iv'].median()
    print(f'Fitted market IV (median): {median_iv:.4f}')

    for strike, voucher_name in vouchers_in_data.items():
        v_sub   = prices_raw[prices_raw['product'] == voucher_name][['abs_ts','mid_price']]\
                    .sort_values('abs_ts').iloc[::10]
        und_sub = prices_raw[prices_raw['product'] == UNDERLYING][['abs_ts','mid_price']]\
                    .sort_values('abs_ts')
        merged  = pd.merge_asof(v_sub, und_sub, on='abs_ts', suffixes=('_opt','_und'))

        elapsed = (merged['abs_ts'] - merged['abs_ts'].min())
        total   = DAYS_TO_EXPIRY_START * TICKS_PER_DAY * 100
        T_years = ((total - elapsed.clip(0, total)) / total) * DAYS_TO_EXPIRY_START / 252

        merged['bs_price'] = [bs_call_price(S, strike, T, median_iv)
                               for S, T in zip(merged['mid_price_und'], T_years)]
        merged['misprice'] = merged['mid_price_opt'] - merged['bs_price']

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7))
        ax1.plot(merged['abs_ts'], merged['mid_price_opt'], label='Market Price', linewidth=0.9)
        ax1.plot(merged['abs_ts'], merged['bs_price'],      label='BS Fair Value (med IV)',
                 linewidth=0.9, linestyle='--')
        ax1.set_title(f'{voucher_name} — Market vs BS Fair Value', fontweight='bold')
        ax1.legend()

        ax2.plot(merged['abs_ts'], merged['misprice'], color='crimson', linewidth=0.9)
        ax2.axhline(0, color='black', linewidth=1)
        sig = merged['misprice'].std()
        ax2.axhline( 2*sig, color='orange', linestyle=':', label='+2σ')
        ax2.axhline(-2*sig, color='orange', linestyle=':', label='-2σ')
        ax2.fill_between(merged['abs_ts'], 2*sig, merged['misprice'].max(),
                          alpha=0.1, color='red', label='Sell zone')
        ax2.fill_between(merged['abs_ts'], merged['misprice'].min(), -2*sig,
                          alpha=0.1, color='green', label='Buy zone')
        ax2.set_title('Mispricing (Market − BS)', fontweight='bold')
        ax2.legend()
        plt.tight_layout()
        plt.show()

    print('\n>> Sell option when misprice > +2σ (market overprices vs BS).\n'
          '>> Buy option when misprice < -2σ (market underprices).\n'
          '>> Delta-hedge simultaneously by trading the underlying in opposite direction.')

---
## Section 7 — Conversion / Tariff Products

**Context:** Products like MAGNIFICENT_MACARONS can be converted between the local market and an external source at a given bid/ask (plus fees). The alpha is: if the internal market price is far enough from the external conversion price (after fees), convert and pocket the difference.

**What to extract:**
- **Import vs export profit** = internal_ask − external_bid − fees (must be < 0 to import profitably)
- **Conversion timing** → look for systematic windows when the opportunity is open

In [ ]:
# ── CONVERSION PRODUCT ANALYSIS ────────────────────────────────────────────────
# The observations data (from log files) contains external bid/ask and fees.
# If you have parsed log files, load them here. Otherwise we show what to look for.

CONV_COLS_EXPECTED = [
    'transportFees', 'exportTariff', 'importTariff', 'sugarPrice',
    'sunlightIndex', 'humidity',      # environmental signals for Macarons
    'bidPrice', 'askPrice',            # external market
]

print('== Conversion / Observations Analysis ==')
print(f'Looking for product: {CONVERSION_PRODUCT}')
print()
print('If you have parsed log files into a DataFrame with these columns:')
print(CONV_COLS_EXPECTED)
print()
print('Plug them in below as `obs_df`. The analysis will then show:')
print('  1. External bid/ask vs internal mid')
print('  2. Import profit = internal_bid - external_ask - importTariff - transportFees')
print('  3. Export profit = external_bid - internal_ask - exportTariff - transportFees')
print('  4. Sugar/sunlight/humidity vs price (environmental alpha)')

# ── TEMPLATE: replace with real obs_df ────────────────────────────────────────
obs_df = pd.DataFrame()  # <-- load your observations DataFrame here

if not obs_df.empty and CONVERSION_PRODUCT in prices_raw.get('product', pd.Series()).values:
    conv_prices = prices_raw[prices_raw['product'] == CONVERSION_PRODUCT][['abs_ts','mid_price',
                                                                             'bid_price_1','ask_price_1']]
    merged = pd.merge_asof(obs_df.sort_values('abs_ts'), conv_prices.sort_values('abs_ts'), on='abs_ts')

    merged['import_profit'] = (merged['bid_price_1']
                                - merged['askPrice']
                                - merged['importTariff']
                                - merged['transportFees'])
    merged['export_profit'] = (merged['bidPrice']
                                - merged['ask_price_1']
                                - merged['exportTariff']
                                - merged['transportFees'])

    fig, axes = plt.subplots(3, 1, figsize=(14, 10))

    axes[0].plot(merged['abs_ts'], merged['mid_price'],   label='Internal Mid')
    axes[0].plot(merged['abs_ts'], merged['bidPrice'],    label='External Bid', linestyle='--')
    axes[0].plot(merged['abs_ts'], merged['askPrice'],    label='External Ask', linestyle=':')
    axes[0].set_title('Internal vs External Price', fontweight='bold')
    axes[0].legend()

    axes[1].plot(merged['abs_ts'], merged['import_profit'], color='green', label='Import Profit')
    axes[1].plot(merged['abs_ts'], merged['export_profit'], color='red',   label='Export Profit')
    axes[1].axhline(0, color='black', linewidth=1)
    axes[1].set_title('Conversion Profitability per Unit (after fees)', fontweight='bold')
    axes[1].legend()

    if 'sugarPrice' in merged.columns:
        ax_env = axes[2]
        ax_env2 = ax_env.twinx()
        ax_env.plot(merged['abs_ts'], merged['sugarPrice'],   color='brown', label='Sugar Price')
        ax_env2.plot(merged['abs_ts'], merged['mid_price'], color='steelblue', alpha=0.5, label='Mid')
        ax_env.set_title('Sugar Price vs Product Mid Price', fontweight='bold')
        ax_env.set_ylabel('Sugar Price', color='brown')
        ax_env2.set_ylabel('Mid Price', color='steelblue')

    plt.suptitle(f'Conversion Analysis — {CONVERSION_PRODUCT}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print('>> When import_profit > 0: convert 10 units (limit) from external to internal per step.\n'
          '>> When export_profit > 0: convert 10 units internal to external.\n'
          '>> Sugar / sunlight / humidity may linearly predict future mid price — run a regression.')

---
## Section 8 — Mean-Reversion & Trend Signals

**What to extract:**
- **Augmented Dickey-Fuller test** → p < 0.05 means the price is stationary (mean-reverting)  
- **Autocorrelation of returns** → negative lag-1 autocorrelation = mean-reverting; positive = trending  
- **Half-life of mean reversion** → how fast prices return to equilibrium (determines holding time)

In [ ]:
# ── STATIONARITY TESTS (ADF) ──────────────────────────────────────────────────
if not prices_raw.empty:
    adf_results = []
    for prod in PRODUCTS:
        sub = prices_raw[prices_raw['product'] == prod]['mid_price'].dropna()
        if len(sub) < 50:
            continue
        stat, pval, lags, nobs, crit, _ = adfuller(sub)
        adf_results.append({
            'product': prod,
            'ADF_stat': stat,
            'p_value': pval,
            'stationary': pval < 0.05,
            'signal': 'MEAN-REVERT' if pval < 0.05 else 'TREND/RANDOM',
            'crit_1%': crit['1%'],
            'crit_5%': crit['5%'],
        })

    adf_df = pd.DataFrame(adf_results).set_index('product')
    print('=== Augmented Dickey-Fuller Stationarity Test ===')
    display(adf_df.round(4))
    print('\n>> Stationary (p < 0.05): trade mean-reversion. Buy when price far below rolling mean,\n'
          '   sell when far above. Optimal entry = μ ± 1.5σ, exit = μ.\n'
          '>> Non-stationary: avoid pure mean-reversion. Use momentum or pairs strategies instead.')

In [ ]:
# ── RETURN AUTOCORRELATION ────────────────────────────────────────────────────
# ACF of returns: negative lag-1 → mean reversion (fade moves)
#                 positive lag-1 → momentum (follow moves)

if not prices_raw.empty:
    n_lags = 30
    fig, axes = plt.subplots(len(PRODUCTS), 2, figsize=(14, 3.5 * len(PRODUCTS)))
    if len(PRODUCTS) == 1:
        axes = [axes]

    for i, prod in enumerate(PRODUCTS):
        sub = prices_raw[prices_raw['product'] == prod].sort_values('abs_ts')
        ret = np.log(sub['mid_price'] / sub['mid_price'].shift(1)).dropna()

        sm.graphics.tsa.plot_acf(ret,  lags=n_lags, ax=axes[i][0], alpha=0.05)
        sm.graphics.tsa.plot_pacf(ret, lags=n_lags, ax=axes[i][1], alpha=0.05)

        lag1 = ret.autocorr(lag=1)
        axes[i][0].set_title(f'{prod} — ACF (lag-1={lag1:.3f})', fontweight='bold')
        axes[i][1].set_title(f'{prod} — PACF', fontweight='bold')

    plt.suptitle('Return Autocorrelation & Partial Autocorrelation', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print('\n>> Bars outside the blue band = statistically significant autocorrelation.\n'
          '>> Negative lag-1 ACF: price tends to reverse the previous tick → market-make aggressively.\n'
          '>> Positive lags at 1-5 ticks: short-term momentum → place resting orders in the direction of move.')

In [ ]:
# ── HALF-LIFE OF MEAN REVERSION ───────────────────────────────────────────────
# Fit: Δprice(t) = λ × (μ − price(t-1)) + ε
# Half-life = −ln(2) / λ (in ticks); tells you how long to hold a mean-reversion trade

if not prices_raw.empty:
    hl_results = []
    for prod in PRODUCTS:
        sub = prices_raw[prices_raw['product'] == prod]['mid_price'].dropna().values
        if len(sub) < 100:
            continue
        delta_p  = np.diff(sub)
        lag_p    = sub[:-1] - sub[:-1].mean()
        X        = sm.add_constant(lag_p)
        res      = sm.OLS(delta_p, X).fit()
        lam      = res.params[1]
        half_life = -np.log(2) / lam if lam < 0 else np.inf

        hl_results.append({
            'product':   prod,
            'lambda':    lam,
            'half_life_ticks': half_life,
            'half_life_seconds': half_life * 0.1 if np.isfinite(half_life) else np.inf,
            'r_squared': res.rsquared,
        })

    hl_df = pd.DataFrame(hl_results).set_index('product')
    print('=== Mean-Reversion Half-Life Estimates ===')
    display(hl_df.round(2))
    print('\n>> Short half-life (< 50 ticks = 5 seconds): very fast reversion → tight market making is ideal.\n'
          '>> Long half-life (> 500 ticks): slow reversion → swing/positional mean-reversion trades.\n'
          '>> Infinite half-life: price is random walk or trending → do NOT mean-revert.')

---
## Section 9 — Cross-Product Correlation & Lead-Lag

**What to extract:**
- **Lead-lag relationship**: if product A moves first and product B follows with a lag, trade B using A as a signal
- **Cointegration**: two products that are individually non-stationary but whose *spread* is stationary → pairs trade
- **Causal direction**: which product "prices in" information first

In [ ]:
# ── LEAD-LAG CROSS-CORRELATION ────────────────────────────────────────────────
MAX_LAG = 50  # ticks

if not prices_raw.empty and len(PRODUCTS) > 1:
    pivot = prices_raw.pivot_table(index='abs_ts', columns='product', values='mid_price', aggfunc='last')
    ret_pivot = np.log(pivot / pivot.shift(1)).fillna(0)

    pairs = [(PRODUCTS[i], PRODUCTS[j])
             for i in range(len(PRODUCTS)) for j in range(i+1, len(PRODUCTS))]

    if len(pairs) > 0:
        n_cols = min(2, len(pairs))
        n_rows = (len(pairs) + n_cols - 1) // n_cols
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
        axes = np.array(axes).flatten()

        lead_lag_results = []
        for idx, (p1, p2) in enumerate(pairs):
            if p1 not in ret_pivot.columns or p2 not in ret_pivot.columns:
                continue
            r1 = ret_pivot[p1].values
            r2 = ret_pivot[p2].values
            lags_range = range(-MAX_LAG, MAX_LAG + 1)
            xcorr = [np.corrcoef(r1[MAX_LAG:], np.roll(r2, lag)[MAX_LAG:])[0,1]
                     for lag in lags_range]

            best_lag = list(lags_range)[np.argmax(np.abs(xcorr))]
            best_corr = max(xcorr, key=abs)
            lead_lag_results.append({'pair': f'{p1}→{p2}', 'best_lag': best_lag,
                                      'best_corr': best_corr})

            if idx < len(axes):
                axes[idx].plot(list(lags_range), xcorr, color=PALETTE[idx % 10], linewidth=1.2)
                axes[idx].axvline(0, color='black', linewidth=0.8, linestyle='--')
                axes[idx].axvline(best_lag, color='red', linewidth=1.2,
                                   linestyle=':', label=f'Best lag={best_lag}')
                axes[idx].set_title(f'{p1} vs {p2}  (max |xcorr| at lag={best_lag})',
                                     fontsize=8, fontweight='bold')
                axes[idx].set_xlabel('Lag (ticks, +ve = p1 leads p2)')
                axes[idx].set_ylabel('Cross-Correlation')
                axes[idx].legend(fontsize=7)

        for j in range(idx + 1, len(axes)):
            axes[j].set_visible(False)

        plt.suptitle(f'Cross-Correlation & Lead-Lag (±{MAX_LAG} ticks)', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()

        print('\n=== Lead-Lag Summary ===')
        display(pd.DataFrame(lead_lag_results))
        print('\n>> Positive lag = p1 leads p2. Use p1\'s return as a signal to trade p2 N ticks later.\n'
              '>> Only act if |best_corr| > 0.05 — weaker signals are noise.')

In [ ]:
# ── COINTEGRATION TEST ────────────────────────────────────────────────────────
# Two non-stationary series can be cointegrated (their linear combination is stationary)
# This is the foundation of pairs trading

if not prices_raw.empty and len(PRODUCTS) > 1:
    pivot_prices = prices_raw.pivot_table(index='abs_ts', columns='product',
                                           values='mid_price', aggfunc='last').ffill()
    coint_results = []

    for i, p1 in enumerate(PRODUCTS):
        for j, p2 in enumerate(PRODUCTS):
            if j <= i:
                continue
            if p1 not in pivot_prices or p2 not in pivot_prices:
                continue
            series = pivot_prices[[p1, p2]].dropna()
            if len(series) < 100:
                continue
            try:
                stat, pval, crit = coint(series[p1], series[p2])
                coint_results.append({
                    'pair': f'{p1} / {p2}',
                    'coint_stat': stat,
                    'p_value': pval,
                    'cointegrated': pval < 0.05,
                    'signal': 'PAIRS TRADE' if pval < 0.05 else '—'
                })
            except Exception as e:
                pass

    if coint_results:
        coint_df = pd.DataFrame(coint_results).set_index('pair')
        print('=== Cointegration Test Results ===')
        display(coint_df.sort_values('p_value').round(4))
        print('\n>> p < 0.05: cointegrated pair. Run OLS to find hedge ratio β:\n'
              '   p1 = α + β × p2  → spread = p1 − β × p2 → trade when spread > ±2σ.\n'
              '>> Basket components with basket are expected to be cointegrated.')

---
## Section 10 — Named Bot Behaviour Deep Dive

**Context:** In Prosperity 3, the bot "Olivia" had a perfectly predictable pattern: buy near the daily low, sell near the daily high. This was one of the highest alpha sources for top teams. Detecting such bots is critical.

**What to extract:**
- **Trade timing pattern** → does the bot always trade at specific timestamps?  
- **Trade price pattern** → does it trade at extremes (daily high/low)?  
- **Consistent size** → same size every time = deterministic bot  
- **Day-over-day consistency** → if pattern repeats, exploit it on day 2+

In [ ]:
# ── BOT TRADE TIMING HEATMAP ──────────────────────────────────────────────────
if not trades_raw.empty and 'buyer' in trades_raw.columns:
    symbol_col = 'symbol' if 'symbol' in trades_raw.columns else 'product'

    for bot in bots_found:
        bot_all = trades_raw[(trades_raw['buyer'] == bot) | (trades_raw['seller'] == bot)].copy()
        if bot_all.empty:
            continue

        bot_all['within_day_ts'] = bot_all['timestamp']   # 0..999_900
        bot_all['ts_bucket'] = (bot_all['within_day_ts'] // 1000) * 1000  # 1000-tick buckets

        prods = bot_all[symbol_col].unique()
        for prod in prods:
            sub = bot_all[bot_all[symbol_col] == prod]
            if sub.empty:
                continue

            fig, axes = plt.subplots(1, 3, figsize=(16, 4))

            # Trade count by time bucket (across all days)
            counts = sub.groupby('ts_bucket').size().reindex(
                range(0, TICKS_PER_DAY * 100 + 1, 1000), fill_value=0)
            axes[0].bar(counts.index, counts.values, width=900, color='royalblue', alpha=0.7)
            axes[0].set_title(f'{bot} — Trade Activity by Timestamp\n({prod})', fontweight='bold')
            axes[0].set_xlabel('Within-Day Timestamp')
            axes[0].set_ylabel('Trade Count')

            # Price vs daily range
            for day in DAYS:
                day_prices = prices_raw[(prices_raw['product'] == prod) &
                                         (prices_raw['source_day'] == day)]['mid_price']
                day_trades = sub[sub['source_day'] == day]
                if day_prices.empty or day_trades.empty:
                    continue
                pct_rank = [(p - day_prices.min()) / (day_prices.max() - day_prices.min() + 1e-9)
                             for p in day_trades['price']]
                axes[1].scatter(day_trades['timestamp'], pct_rank,
                                 label=f'Day {day}', s=15, alpha=0.7)
            axes[1].axhline(0.0, color='green', linestyle='--', linewidth=1, label='Daily Low')
            axes[1].axhline(1.0, color='red',   linestyle='--', linewidth=1, label='Daily High')
            axes[1].set_title(f'{bot} — Trade Price Percentile in Daily Range\n({prod})', fontweight='bold')
            axes[1].set_xlabel('Timestamp')
            axes[1].set_ylabel('Price % of Daily Range')
            axes[1].legend(fontsize=7)

            # Trade size distribution
            axes[2].hist(sub['quantity'].abs(), bins=20, color='darkorange', alpha=0.8)
            axes[2].set_title(f'{bot} — Order Size Distribution\n({prod})', fontweight='bold')
            axes[2].set_xlabel('Trade Size')
            axes[2].set_ylabel('Frequency')

            plt.suptitle(f'Bot Deep Dive: {bot} — {prod}', fontsize=12, fontweight='bold')
            plt.tight_layout()
            plt.show()

    print('\n>> If a bot trades at the same timestamps with the same sizes day after day:\n'
          '   → Guaranteed alpha. Position BEFORE its expected trade time.\n'
          '>> If price percentile of trades is always near 0 (buys at low) / 1 (sells at high):\n'
          '   → The bot has perfect daily-range timing. Buy before it buys, sell before it sells.')

In [ ]:
# ── DAY-OVER-DAY BOT PATTERN CONSISTENCY ─────────────────────────────────────
# Do bots repeat their exact behaviour? Compare trade timestamps across days.

if not trades_raw.empty and 'buyer' in trades_raw.columns and len(DAYS) > 1:
    symbol_col = 'symbol' if 'symbol' in trades_raw.columns else 'product'

    for bot in bots_found:
        bot_all = trades_raw[(trades_raw['buyer'] == bot) | (trades_raw['seller'] == bot)].copy()
        if bot_all.empty or len(bot_all['source_day'].unique()) < 2:
            continue

        for prod in bot_all[symbol_col].unique():
            sub = bot_all[bot_all[symbol_col] == prod]
            day_ts = {day: sorted(sub[sub['source_day'] == day]['timestamp'].tolist())
                       for day in DAYS if day in sub['source_day'].values}

            if len(day_ts) < 2:
                continue

            fig, ax = plt.subplots(figsize=(14, 3))
            for d, color in zip(sorted(day_ts), ['blue','orange','green']):
                ax.scatter(day_ts[d], [d] * len(day_ts[d]), color=color, s=20,
                            label=f'Day {d}', zorder=5)
            ax.set_title(f'{bot} — {prod}: Trade Timestamps by Day', fontweight='bold')
            ax.set_xlabel('Within-Day Timestamp')
            ax.set_ylabel('Day')
            ax.legend()
            plt.tight_layout()
            plt.show()

    print('\n>> If dots align vertically across days → bot trades at EXACTLY the same time daily.\n'
          '>> Use day -2 data to predict day -1 behaviour; use days -2 and -1 to front-run day 0.')

---
## Section 11 — PnL Attribution & Position Analysis

**What to extract:**
- **Cumulative PnL per product** → see which products are driving (or destroying) your returns  
- **PnL drawdown** → maximum loss from peak; helps size risk limits per product  
- **PnL slope** → is each product's PnL growing consistently or erratically?

In [ ]:
# ── CUMULATIVE PnL BY PRODUCT ─────────────────────────────────────────────────
if not prices_raw.empty and 'profit_and_loss' in prices_raw.columns:
    pnl_data = prices_raw.groupby(['abs_ts', 'product'])['profit_and_loss'].last().unstack(fill_value=np.nan)

    fig, ax = plt.subplots(figsize=(14, 5))
    for i, prod in enumerate(pnl_data.columns):
        series = pnl_data[prod].ffill()
        ax.plot(pnl_data.index, series, label=prod, linewidth=1.2, alpha=0.85)

    ax.set_title('Cumulative PnL by Product', fontweight='bold', fontsize=13)
    ax.set_xlabel('Absolute Timestamp')
    ax.set_ylabel('PnL (seashells)')
    ax.legend(ncol=3, fontsize=8)
    ax.axhline(0, color='black', linewidth=0.8)
    plt.tight_layout()
    plt.show()

    # Summary stats
    pnl_last = pnl_data.ffill().iloc[-1].sort_values(ascending=False)
    print('=== Final PnL by Product ===')
    for p, v in pnl_last.items():
        print(f'  {p:35s}: {v:>12,.2f}')
    print(f'\n  TOTAL: {pnl_last.sum():,.2f}')

In [ ]:
# ── PnL DRAWDOWN ──────────────────────────────────────────────────────────────
def max_drawdown(series):
    running_max = series.cummax()
    drawdown = series - running_max
    return drawdown

if not prices_raw.empty and 'profit_and_loss' in prices_raw.columns:
    pnl_data = prices_raw.groupby(['abs_ts', 'product'])['profit_and_loss'].last().unstack(fill_value=np.nan)

    n_prod = len(pnl_data.columns)
    fig, axes = plt.subplots(n_prod, 1, figsize=(14, 3 * n_prod))
    if n_prod == 1:
        axes = [axes]

    dd_summary = []
    for i, prod in enumerate(pnl_data.columns):
        series = pnl_data[prod].ffill().dropna()
        dd = max_drawdown(series)
        max_dd = dd.min()
        dd_summary.append({'product': prod, 'max_drawdown': max_dd,
                            'final_pnl': series.iloc[-1]})

        ax = axes[i]
        ax.fill_between(dd.index, dd.values, 0, color='red', alpha=0.4, label='Drawdown')
        ax.plot(series.index, series, color='blue', linewidth=0.8, alpha=0.7, label='PnL')
        ax.set_title(f'{prod} — PnL & Drawdown (max DD = {max_dd:,.0f})', fontweight='bold')
        ax.legend(fontsize=7)

    plt.suptitle('PnL Drawdown by Product', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    dd_df = pd.DataFrame(dd_summary).set_index('product')
    display(dd_df.round(2))
    print('\n>> Products with large drawdown relative to final PnL: risk/reward is poor.\n'
          '>> Tighten position limits or add a stop-loss threshold for these products.')

---
## Section 12 — Alpha Signal Summary Dashboard

All signals computed above, consolidated into a single decision table.

In [ ]:
# ── SIGNAL SCORECARD ──────────────────────────────────────────────────────────
import textwrap

signal_rows = []

if not prices_raw.empty:
    for prod in PRODUCTS:
        sub = prices_raw[prices_raw['product'] == prod]['mid_price'].dropna()
        if len(sub) < 50:
            continue

        ret = np.log(sub / sub.shift(1)).dropna()

        # Stationarity
        _, adf_p, *_ = adfuller(sub)
        mean_rev = adf_p < 0.05

        # Half-life
        delta_p = np.diff(sub.values)
        lag_p   = sub.values[:-1] - sub.values[:-1].mean()
        X       = sm.add_constant(lag_p)
        res     = sm.OLS(delta_p, X).fit()
        lam     = res.params[1]
        hl      = -np.log(2) / lam if lam < 0 else np.inf

        # Return autocorrelation
        lag1_ac = ret.autocorr(lag=1)

        # Spread
        spread_med = prices_raw[prices_raw['product'] == prod].eval('ask_price_1 - bid_price_1').median()

        # OBI correlation
        obi_corr = np.nan
        if all(c in prices_raw.columns for c in ['bid_volume_1','ask_volume_1']):
            s2 = prices_raw[prices_raw['product'] == prod].copy()
            bv = s2['bid_volume_1'].abs(); av = s2['ask_volume_1'].abs()
            s2['obi'] = (bv - av) / (bv + av).replace(0, np.nan)
            s2['fwd'] = s2['mid_price'].shift(-10) / s2['mid_price'] - 1
            obi_corr = s2[['obi','fwd']].dropna().corr().iloc[0,1]

        strategy = []
        if mean_rev and np.isfinite(hl) and hl < 200:
            strategy.append('MarketMake/MeanRev')
        if lag1_ac > 0.05:
            strategy.append('Momentum')
        if not mean_rev and np.isinf(hl):
            strategy.append('Trend-follow')
        if abs(obi_corr) > 0.05 if not np.isnan(obi_corr) else False:
            strategy.append('OBI-signal')
        if not strategy:
            strategy.append('—')

        signal_rows.append({
            'product':          prod,
            'stationary(p)':    f'{adf_p:.3f}',
            'half_life(ticks)': f'{hl:.0f}' if np.isfinite(hl) else '∞',
            'lag1_autocorr':    f'{lag1_ac:.3f}',
            'median_spread':    f'{spread_med:.1f}',
            'obi_fwd_corr':     f'{obi_corr:.3f}' if not np.isnan(obi_corr) else 'N/A',
            'suggested_strategy': ' + '.join(strategy),
        })

scorecard = pd.DataFrame(signal_rows).set_index('product')
print('=' * 80)
print('ALPHA SIGNAL SCORECARD')
print('=' * 80)
display(scorecard)

print('\nLEGEND')
print('  stationary(p) < 0.05  → price is mean-reverting (ADF test)')
print('  half_life(ticks)       → how fast price reverts (low = fast = MM/MeanRev)')
print('  lag1_autocorr > 0      → positive momentum in returns')
print('  lag1_autocorr < 0      → negative autocorr (mean-reverting returns, good for MM)')
print('  median_spread          → bid-ask spread; your edge must exceed half this')
print('  obi_fwd_corr > 0.05   → OBI is a directional signal (trade in OBI direction)')
print('  suggested_strategy     → primary alpha to pursue for each product')

In [ ]:
# ── INTERACTIVE PRICE EXPLORER (Plotly) ──────────────────────────────────────
# Pan and zoom to inspect any time window

if not prices_raw.empty:
    fig = make_subplots(
        rows=len(PRODUCTS), cols=1,
        shared_xaxes=True,
        subplot_titles=[f'{p} — Mid Price' for p in PRODUCTS],
        vertical_spacing=0.04,
    )
    for i, prod in enumerate(PRODUCTS, start=1):
        sub = prices_raw[prices_raw['product'] == prod].sort_values('abs_ts')
        fig.add_trace(
            go.Scatter(x=sub['abs_ts'], y=sub['mid_price'], mode='lines',
                       name=prod, line=dict(width=1)),
            row=i, col=1
        )
        if not trades_raw.empty:
            sc = 'symbol' if 'symbol' in trades_raw.columns else 'product'
            t_prod = trades_raw[trades_raw[sc] == prod]
            if not t_prod.empty:
                fig.add_trace(
                    go.Scatter(x=t_prod['abs_ts'], y=t_prod['price'],
                               mode='markers', name=f'{prod} trades',
                               marker=dict(size=4, opacity=0.5, color='black')),
                    row=i, col=1
                )

    fig.update_layout(
        height=300 * len(PRODUCTS),
        title='Interactive Price Explorer — All Products (with Trades)',
        showlegend=True
    )
    fig.show()
    print('>> Use the Plotly toolbar to zoom in on specific windows.\n'
          '>> Black dots = market trades; use them to identify bot activity clusters.')

In [ ]:
# ── STRATEGY IMPLEMENTATION TEMPLATES ─────────────────────────────────────────
print("""
================================================================================
STRATEGY TEMPLATES — paste into your trader.py
================================================================================

1. PURE MARKET MAKING (for stationary, low-spread products)
   fair_value = rolling_ema(mid_price, span=20)
   buy_order  = fair_value - spread/2  (or better)
   sell_order = fair_value + spread/2  (or better)
   ── Width should be at least 1 tick above median spread.

2. MEAN-REVERSION (for stationary products with half-life < 200 ticks)
   z_score = (current_price - rolling_mean) / rolling_std
   if z_score > 1.5: sell to short, target mean
   if z_score < -1.5: buy long, target mean
   exit when |z_score| < 0.3

3. BASKET ARBITRAGE (for PICNIC_BASKET products)
   fair = sum(qty_i * mid_price_i for each component)
   spread = basket_mid - fair
   if spread > +threshold: sell basket, buy components
   if spread < -threshold: buy basket, sell components
   threshold ≥ transaction_cost + 1 tick

4. OPTIONS (VOLCANIC_ROCK_VOUCHER)
   iv_market = implied_vol(market_price, S, K, T)
   iv_fair   = smile_curve_fit(K / S)   # fit a quadratic to all strikes
   if iv_market < iv_fair - 0.02: BUY call + delta-hedge (sell underlying)
   if iv_market > iv_fair + 0.02: SELL call + delta-hedge (buy underlying)

5. BOT-FOLLOWING (for Olivia/Vinnie patterns)
   if bot bought yesterday at timestamp T → pre-position long before T today
   if bot sold yesterday at timestamp T  → pre-position short before T today
   Use tight limits — bots can shift behaviour round-to-round.

6. OBI SIGNAL
   obi = (bid_vol_1 - ask_vol_1) / (bid_vol_1 + ask_vol_1)
   if obi > 0.5: submit market buy (fill against thin ask)
   if obi < -0.5: submit market sell
   ── Only use when obi_fwd_corr > 0.05 from scorecard above.
================================================================================
""")

---
## Appendix — Utilities

Reusable helper functions referenced above, collected for convenience.

In [ ]:
# ── GENERAL UTILITIES ─────────────────────────────────────────────────────────

def rolling_zscore(series, window=100):
    """Rolling z-score for mean-reversion signals."""
    m = series.rolling(window).mean()
    s = series.rolling(window).std().replace(0, np.nan)
    return (series - m) / s


def ema_fair_value(series, span=20):
    """Exponential moving average as fair value estimator."""
    return series.ewm(span=span, adjust=False).mean()


def vwap(price_series, volume_series):
    """Volume-weighted average price."""
    return (price_series * volume_series).cumsum() / volume_series.cumsum()


def compute_position_pnl(price_series, position_series):
    """Mark-to-market PnL from a position series."""
    price_chg = price_series.diff().fillna(0)
    return (position_series.shift(1).fillna(0) * price_chg).cumsum()


def fit_vol_smile(strikes, ivs, degree=2):
    """Fit a polynomial to the IV smile; return fitted function."""
    coeffs = np.polyfit(strikes, ivs, degree)
    return np.poly1d(coeffs)


def sharpe(returns, freq=TICKS_PER_DAY):
    """Annualised Sharpe ratio."""
    mu = returns.mean()
    sd = returns.std()
    if sd == 0:
        return np.nan
    return mu / sd * np.sqrt(freq)


print('Utility functions ready:')
print('  rolling_zscore, ema_fair_value, vwap, compute_position_pnl,')
print('  fit_vol_smile, sharpe, bs_call_price, implied_vol, bs_delta')